In [1]:
import numpy as np
import pandas as pd

In [6]:
movies  = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [7]:
movies.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


### Data Preprocessing

In [8]:
# merge both dataframes in movies
movies = movies.merge(credits, on='title')

In [9]:
# keeping only necessary columns out of 23 for recommendations
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

In [10]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [11]:
# checks for all the columns if there exists null values and sum gives the num of null values
movies.isnull().sum()

,0
movie_id,0
title,0
overview,3
genres,0
keywords,0
cast,0
crew,0


In [12]:
movies.dropna(inplace=True) # dropped the rows(movies) where the column is null

In [13]:
movies.duplicated().sum()

0

In [14]:
movies.iloc[0].genres # output is string

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [15]:
import ast

In [16]:
# we have to keep only tags 'action', 'adventure'...
# so we need to convert this output to list containing these tags
def convert(obj):
  L=[]
  for i in ast.literal_eval(obj):
    L.append(i['name'])
  return L


In [17]:
# convert the genres column datatype to list
movies['genres'] = movies['genres'].apply(convert)

In [18]:
movies['keywords'] = movies['keywords'].apply(convert)

In [19]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [20]:
movies['cast']

,cast
0,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""..."
1,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa..."
2,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr..."
3,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba..."
4,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c..."
...,...
4804,"[{""cast_id"": 1, ""character"": ""El Mariachi"", ""c..."
4805,"[{""cast_id"": 1, ""character"": ""Buzzy"", ""credit_..."
4806,"[{""cast_id"": 8, ""character"": ""Oliver O\u2019To..."
4807,"[{""cast_id"": 3, ""character"": ""Sam"", ""credit_id..."


In [21]:
 # we have to keep only top three actors in cast
def convert_cast(obj):
  L = []
  count = 0
  for i in ast.literal_eval(obj):
    if count !=3:
      L.append(i['name'])
      count += 1
    else:
      break
  return L

In [22]:
movies['cast'] = movies['cast'].apply(convert_cast)

In [23]:
# from the crew we want only director
def fetch_director(obj):
  L = []
  for i in ast.literal_eval(obj):
    if(i['job']=='Director'):
      L.append(i['name'])
      break
  return L

In [24]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [25]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [26]:
movies['overview'][0]

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

In [27]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())


In [28]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [29]:
# now we are removing any spaces between the words so that it remains a single entity
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ", "") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ", "") for i in x])
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]


In [30]:
# tags for a movie will be a paragraph containing overview, genres, keywords, cast, crew
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [31]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],"[In, the, 22nd, century,, a, paraplegic, Marin..."


In [32]:
new_df = movies[['movie_id', 'title', 'tags']]

In [33]:
# convert the tags list to string i.e, joining elements of list and separate them by whitespace
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

<ipython-input-33-2170ccf6c194>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [34]:
new_df.head(1)

,movie_id,title,tags
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di..."


In [35]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

<ipython-input-35-db70cfe1859e>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [36]:
new_df.head(1)

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."


### Vectorization


In [37]:
# now tags contains words which are almost same such as actor, actors . So we perform stemming that will make these words same actor, actor
import nltk   #natural language toolkit


In [38]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [39]:
def stem(text):
  y = []
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)  # join the stemmed words in the list to form a string of words separated by whitespaces

In [40]:
stem(new_df['tags'][0]) # you can see the stemmed words

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi sciencefict cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d samworthington zoesaldana sigourneyweav jamescameron'

In [41]:
new_df['tags'] = new_df['tags'].apply(stem)

<ipython-input-41-be18a4346d89>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


In [42]:
# We will make vectors which will represent movies
from sklearn.feature_extraction.text import CountVectorizer
# remove the stopwords from the text and then form vectors by comparing 5000 words
cv = CountVectorizer(max_features=5000, stop_words='english')

In [43]:
vectors = cv.fit_transform(new_df['tags']).toarray()

In [44]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0])

In [45]:
cv.get_feature_names_out() # these are the most appeared 5000 words

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [46]:
# let's see what ps.stem() does
ps.stem('acting')

'act'

In [47]:
vectors.shape

(4806, 5000)

### Check for closest vectors

Now we will check similarity of one vector with all other vectors based on cosine distance instead of Euclidian distance as it fails for higher dimensions.

In [48]:
from sklearn.metrics.pairwise import cosine_similarity

In [49]:
similarity = cosine_similarity(vectors)  # outputs the cosine distances of every movie with every other movies

In [50]:
similarity

array([[1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
        0.        ],
       [0.08346223, 1.        , 0.06063391, ..., 0.02378257, 0.        ,
        0.02615329],
       [0.0860309 , 0.06063391, 1.        , ..., 0.02451452, 0.        ,
        0.        ],
       ...,
       [0.04499213, 0.02378257, 0.02451452, ..., 1.        , 0.03962144,
        0.04229549],
       [0.        , 0.        , 0.        , ..., 0.03962144, 1.        ,
        0.08714204],
       [0.        , 0.02615329, 0.        , ..., 0.04229549, 0.08714204,
        1.        ]])

### Recommend 5 most simliar movies

In [51]:
new_df[new_df['title']=='Batman Begins'].index[0] # get the index of a particular movie

119

We wil now for a given movie fetch its index and then based on that index from the similarity matrix we will fetch its distances from all other movies. Then sort those distances and return top 5. But simply sorting will lose the indices of the movies. So we will convert the distances list to tuple(index, distance) and then sort on the basis of distances.

In [52]:
sorted(list(enumerate(similarity[0])), reverse=True, key = lambda x:x[1])[1:6]  # this is how it will work

[(1216, 0.28676966733820225),
 (2409, 0.26901379342448517),
 (3730, 0.2605130246476754),
 (507, 0.255608593705383),
 (539, 0.25038669783359574)]

In [53]:
def recommend(movie):
  movie_index = new_df[new_df['title']==movie].index[0]
  distances = similarity[movie_index]
  movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1])[1:6]

  for i in movies_list:
    print(new_df.iloc[i[0]].title)

In [54]:
recommend('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


### Transfer this file to another platform

In [55]:
import joblib

Creates a .pkl file and dump it in the project folder

In [56]:
joblib.dump(new_df, 'movies.pkl')

['movies.pkl']

In [57]:
joblib.dump(similarity, 'similarity.pkl')

['similarity.pkl']